# Shadow-Net · SOREL 7M — Entrenamiento MLP (notebook canonico)
Master — notebook DEFINITIVO de entrenamiento. Contrato inviolable: `X = [EMBER_2381 | OVERLAY_N]` — las 2381 features EMBER nunca se reordenan, eliminan ni modifican su semantica; overlay solo como apendice (fase futura).

Mapa de fases: **0 Environment (impl), 1 Data access (impl), 2 Dataset manifest (impl, regenera determinista seed 42), 3 StandardScaler 2381 (impl)**, 4 Overlay (RESERVADA), 5 Dataset/DataLoader (RESERVADA), 6 FFNN (RESERVADA), 7 Training (RESERVADA), 8 Validation/metrics (RESERVADA), 9 Ablation/threshold (RESERVADA), 10 ONNX export (RESERVADA).


In [ ]:
# FASE 0 — Environment / reproducibilidad (solo observa y reporta).
import os, sys, shutil, subprocess, platform
GLOBAL_SEED = 42
import numpy as np
np.random.seed(GLOBAL_SEED)

def sh(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, timeout=15).strip()
    except Exception as e:
        return f"NA ({e})"

print("python:", sys.version.split()[0], "|", platform.platform())
print("numpy:", np.__version__)
try:
    import sklearn
    print("sklearn:", sklearn.__version__)
except ImportError:
    print("sklearn: NO INSTALADO")
try:
    import pandas
    print("pandas:", pandas.__version__)
except ImportError:
    print("pandas: NO INSTALADO")
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), end="")
    if torch.cuda.is_available():
        print(f" | {torch.cuda.get_device_name(0)} | VRAM={torch.cuda.get_device_properties(0).total_memory/2**30:.1f}GB")
    else:
        print(" | CPU-only en esta fase (scaler es CPU)")
except ImportError:
    print("torch: no instalado (se necesitara desde Fase 5)")
print("nvidia-smi:", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>&1 | head -5"))
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"RAM total={vm.total/2**30:.1f}GB disponible={vm.available/2**30:.1f}GB")
except ImportError:
    print("psutil: no instalado")
print("df /kaggle/working:", sh("df -h /kaggle/working 2>&1 | tail -1"))
print("df /tmp:", sh("df -h /tmp 2>&1 | tail -1"))
print(f"GLOBAL_SEED={GLOBAL_SEED}")
print("PASS Fase 0: entorno caracterizado.")


In [ ]:
# FASE 1 — Data access: Range/ZIP64 sobre train-features.npz (sin descargar 121GB).
# Reutiliza la logica validada H1-H4: EOCD clasico -> ZIP64 -> Central Directory ->
# Local File Header (STORED=0) -> header .npy -> PAY0/payload. Termina con asserts duros.
import os, struct, time, urllib.request, urllib.error
import numpy as np

NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"
TRAIN_SPLIT = 1543542570.0  # config.py oficial sophos/SOREL-20M
VAL_SPLIT = 1547279640.0
EXPECTED_NPZ_ROWS = 12699013
EXPECTED_NPZ_COLS = 2381
N_FEATURES = 2381
ROW_BYTES = 2381 * 4  # float32 STORED sin compresion
META_URL = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/processed-data/meta.db"
META_SIZE = 3788979200
MISSING_URL = "https://github.com/sophos/SOREL-20M/raw/master/shas_missing_ember_features.json"
SKIP_VALIDATED = True
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
META = os.path.join(WORK, "meta.db")
MISS = os.path.join(WORK, "shas_missing_ember_features.json")
PACE = 0.35

def npz_range(a, b, tries=8, timeout=120):
    # GET con reintentos: tolera 416/503/429/500 (throttling S3), aborta ante otros.
    for k in range(tries):
        try:
            time.sleep(PACE if k == 0 else 8 * k)
            req = urllib.request.Request(NPZ, headers={"Range": f"bytes={a}-{b}"})
            r = urllib.request.urlopen(req, timeout=timeout)
            assert r.status == 206, f"Range no honrado: {r.status}"
            d = r.read()
            assert len(d) == b - a + 1, f"truncado: {len(d)} vs {b-a+1}"
            return d
        except urllib.error.HTTPError as e:
            if e.code in (416, 503, 429, 500):
                continue
            raise
    raise RuntimeError(f"npz_range agotado {a}-{b}")

TOTAL = 121046992510  # verificado H2 via Content-Range
tail = npz_range(TOTAL - 131072, TOTAL - 1)
eocd = tail.rfind(b"PK\x05\x06")
assert eocd != -1, "FAIL: EOCD no encontrado"
(n_disk, n_cd, n_entries, n_total, cd_size, cd_off, _) = struct.unpack("<HHHHIIH", tail[eocd+4:eocd+22])
if cd_off == 0xFFFFFFFF:  # ZIP64: offset real de 64 bits
    loc = tail.rfind(b"PK\x06\x07")
    assert loc != -1, "FAIL: sin locator ZIP64"
    (_, zoff, _) = struct.unpack("<IQI", tail[loc+4:loc+20])
    z = npz_range(zoff, zoff + 55)
    assert z[:4] == b"PK\x06\x06", "FAIL: firma ZIP64 EOCD ausente"
    (_, _, _, _, _, _, n_total, cd_size, cd_off) = struct.unpack("<QHHIIQQQQ", z[4:56])
assert n_total >= 1 and cd_size > 0, "FAIL: inventario vacio"
cd_blob = npz_range(cd_off, cd_off + cd_size - 1)
entries, pos = [], 0
while pos < len(cd_blob):
    assert cd_blob[pos:pos+4] == b"PK\x01\x02", f"FAIL: firma CD corrupta en {pos}"
    (vneed, flags, comp, mt, md, crc, csz32, usz32, nlen, elen, clen, disk, iattr, eattr, lho32) = struct.unpack("<HHHHHIIIHHHHHII", cd_blob[pos+6:pos+46])
    name = cd_blob[pos+46:pos+46+nlen].decode()
    extra = cd_blob[pos+46+nlen:pos+46+nlen+elen]
    csize, usize, lho = csz32, usz32, lho32
    j = 0  # extra ZIP64 id 0x0001 rellena campos 0xFFFFFFFF en orden usize/csize/lho
    while j + 4 <= len(extra):
        hid, dsz = struct.unpack("<HH", extra[j:j+4])
        if hid == 0x0001:
            nq = dsz // 8
            vals = struct.unpack("<" + "Q" * nq, extra[j+4:j+4+8*nq])
            vi = 0
            if usize == 0xFFFFFFFF: usize = vals[vi]; vi += 1
            if csize == 0xFFFFFFFF: csize = vals[vi]; vi += 1
            if lho == 0xFFFFFFFF: lho = vals[vi]; vi += 1
            break
        j += 4 + dsz
    entries.append({"name": name, "compress": comp, "csize": csize, "usize": usize, "lho": lho})
    pos += 46 + nlen + elen + clen
arr = [e for e in entries if e["name"].endswith(".npy")]
assert arr, "FAIL: sin entradas .npy"
assert all(e["compress"] == 0 for e in arr), "FAIL: DEFLATE — row-Range imposible"
lho = arr[0]["lho"]
lh = npz_range(lho, lho + 29)
assert lh[:4] == b"PK\x03\x04", "FAIL: firma Local File Header ausente"
assert struct.unpack("<H", lh[8:10])[0] == 0, "FAIL: arr_0 no STORED"
nlen = struct.unpack("<H", lh[26:28])[0]
elen = struct.unpack("<H", lh[28:30])[0]
data_off = lho + 30 + nlen + elen
npy_head = npz_range(data_off, data_off + 127)
assert npy_head[:6] == b"\x93NUMPY", "FAIL: magia NPY ausente"
hlen = struct.unpack("<H", npy_head[8:10])[0]
hdr = npz_range(data_off, data_off + 10 + hlen - 1)[10:].decode("latin1")
d = eval(hdr)  # formato controlado por numpy; magia ya validada
shape, DT0, order = tuple(d["shape"]), np.dtype(d["descr"]), d["fortran_order"]
assert order is False, "FAIL: array Fortran"
assert shape == (EXPECTED_NPZ_ROWS, EXPECTED_NPZ_COLS), f"FAIL: shape={shape}"
assert DT0 == np.dtype("<f4"), f"FAIL: dtype={DT0}"
PAY0 = data_off + 10 + hlen  # offset del primer float32 de arr_0
print(f"arr_0: shape={shape} dtype={DT0} PAY0={PAY0} ROW_BYTES={ROW_BYTES}")
print("PASS Fase 1: acceso Range/ZIP64 verificado, arr_0 STORED localizable por fila.")


In [ ]:
t0_phase2 = None
# CELDA 2 — Descarga meta.db (única descarga pesada: 3.8GB) + shas_missing (KB/MB).
# Hipótesis: sqlite exige fichero local; no hay Range posible. Con reanudación y chequeo de tamaño.
import os, urllib.request, json
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
META = os.path.join(WORK, "meta.db")
have = os.path.getsize(META) if os.path.exists(META) else 0
print("meta.db presente:", have, "/", META_SIZE)
if have != META_SIZE:
    print("descargando meta.db ...")
    req = urllib.request.Request(META_URL)
    if have > 0:
        req.add_header("Range", "bytes=%d-" % have)
    r = urllib.request.urlopen(req, timeout=120)
    print("http:", r.status)
    mode = "ab" if have > 0 and r.status == 206 else "wb"
    f = open(META, mode)
    done = os.path.getsize(META) if mode == "ab" else 0
    while True:
        b = r.read(8*2**20)
        if not b: break
        f.write(b); done += len(b)
        if (done // 2**30) != ((done - len(b)) // 2**30): print(" ...", done/2**30, "GB")
    f.close()
    assert os.path.getsize(META) == META_SIZE, "FAIL: meta.db incompleto"
print("PASS celda 2a: meta.db completo.")
MISS = os.path.join(WORK, "shas_missing_ember_features.json")
if not os.path.exists(MISS):
    urllib.request.urlretrieve(MISSING_URL, MISS)
missing = set(json.load(open(MISS)))
print("PASS celda 2b: missing set =", len(missing))


In [ ]:
SHA_ORDER = os.path.join(WORK, "train_sha_order.npy")
LAB_SURV = os.path.join(WORK, "train_lab_surv.npy")
SKIP_BUILD = SKIP_VALIDATED and os.path.exists(SHA_ORDER) and os.path.exists(LAB_SURV)
if SKIP_BUILD:
    print("artefactos existentes: se omite reconstrucción del orden train.")
else:
    # ESTRUCTURAL H6: el npz guarda vectores RAW y en orden lexicográfico de sha
    # (demostrado: fila 0 == clave LMDB mínima, fila N-1 == máxima). El rowid puro falla.
    import sqlite3, numpy as np
    con = sqlite3.connect(META)
    n_train = con.execute("SELECT COUNT(*) FROM meta WHERE rl_fs_t <= ?", (TRAIN_SPLIT,)).fetchone()[0]
    print("filas train oficial:", n_train)
    ORDER = os.path.join(WORK, "train_all_order.npy")
    mm = np.memmap(ORDER, dtype="S64", mode="w+", shape=(n_train,))
    pos = 0
    for (sha,) in con.execute("SELECT sha256 FROM meta WHERE rl_fs_t <= ? ORDER BY rowid", (TRAIN_SPLIT,)):
        mm[pos] = sha.encode(); pos += 1
        if pos % 2000000 == 0: print(" ...", pos)
    mm.flush()
    assert pos == n_train
    print("memmap train completo:", mm.shape)
    miss_arr = np.array(sorted(missing), dtype="S64")
    keep = np.ones(n_train, dtype=bool)
    CH = 1000000
    for a in range(0, n_train, CH):
        keep[a:a+CH] = ~np.isin(mm[a:a+CH], miss_arr)
    n_keep = int(keep.sum())
    print("supervivientes:", n_keep, "esperado:", EXPECTED_NPZ_ROWS)
    assert n_keep == EXPECTED_NPZ_ROWS, "FAIL H6 estructural: missing no reproduce filas"
    # supervivientes en orden rowid y reordenado lexicográficamente
    surv_unsorted = np.memmap(os.path.join(WORK, "train_sha_unsorted.npy"), dtype="S64", mode="w+", shape=(n_keep,))
    w = 0
    for a in range(0, n_train, CH):
        blk = mm[a:a+CH][keep[a:a+CH]]
        surv_unsorted[w:w+len(blk)] = blk; w += len(blk)
    surv_unsorted.flush()
    order = np.argsort(surv_unsorted)
    surv = np.memmap(SHA_ORDER, dtype="S64", mode="w+", shape=(n_keep,))
    # copiado por chunks para no duplicar 800MB en RAM
    CH2 = 2000000
    for a in range(0, n_keep, CH2):
        surv[a:a+CH2] = surv_unsorted[order[a:a+CH2]]
    surv.flush()
    print("PASS H6 estructural: orden SORTED. Artefacto:", SHA_ORDER)
    # labels en el mismo orden sorted
    labmm = np.memmap(os.path.join(WORK, "train_lab_all.npy"), dtype="i1", mode="w+", shape=(n_train,))
    pos = 0
    for (m,) in con.execute("SELECT is_malware FROM meta WHERE rl_fs_t <= ? ORDER BY rowid", (TRAIN_SPLIT,)):
        labmm[pos] = m; pos += 1
    labmm.flush()
    lab_filt = np.memmap(os.path.join(WORK, "train_lab_filt.npy"), dtype="i1", mode="w+", shape=(n_keep,))
    w = 0
    for a in range(0, n_train, CH):
        blk = labmm[a:a+CH][keep[a:a+CH]]
        lab_filt[w:w+len(blk)] = blk; w += len(blk)
    lab_filt.flush()
    labsurv = np.memmap(LAB_SURV, dtype="i1", mode="w+", shape=(n_keep,))
    for a in range(0, n_keep, CH2):
        labsurv[a:a+CH2] = lab_filt[order[a:a+CH2]]
    labsurv.flush()
    print("mapa labels sorted:", labsurv.shape, "positivos:", int(labsurv.sum()), "tasa:", float(labsurv.mean()))
    con.close()


In [ ]:
# FASE 2A — Inventario TRAIN antes de seleccionar (sin asumir 50/50)
import os, sqlite3, numpy as np, json, time, psutil
t0 = time.time()
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
SHA_ORDER = os.path.join(WORK, "train_sha_order.npy")
LAB_SURV = os.path.join(WORK, "train_lab_surv.npy")
META = os.path.join(WORK, "meta.db")
MISS = os.path.join(WORK, "shas_missing_ember_features.json")
TRAIN_SPLIT = 1543542570.0
VAL_SPLIT = 1547279640.0
EXPECTED_NPZ_ROWS = 12699013

# Verificar artefactos validados
assert os.path.exists(SHA_ORDER) and os.path.exists(LAB_SURV), "Falta train_sha_order/lab_surv — Fase 1 no completada"
assert os.path.exists(META), "Falta meta.db"
missing = set(json.load(open(MISS)))
print(f"Artefacto SHA_ORDER: {os.path.getsize(SHA_ORDER)/2**20:.1f} MB")
print(f"Artefacto LAB_SURV: {os.path.getsize(LAB_SURV)/2**20:.1f} MB")

# Carga memmap validada (SORTED)
shas = np.memmap(SHA_ORDER, dtype="S64", mode="r")
labs = np.memmap(LAB_SURV, dtype="i1", mode="r")
assert shas.shape[0]==EXPECTED_NPZ_ROWS and labs.shape[0]==EXPECTED_NPZ_ROWS
n_mal = int((labs==1).sum()); n_ben = int((labs==0).sum())
print(f"Candidatos TRAIN válidos (SORTED, sin missing): {shas.shape[0]}")
print(f"  malware: {n_mal} ({n_mal/shas.shape[0]:.2%})")
print(f"  benigno: {n_ben} ({n_ben/shas.shape[0]:.2%})")
print(f"  missing EMBER excluidos: {len(missing)} (verificado estructural)")
# timestamps: construir artefacto alineado con SORTED si no existe
TS_SORTED = os.path.join(WORK, "train_ts_sorted.npy")
if not os.path.exists(TS_SORTED):
    print("Construyendo train_ts_sorted.npy alineado con SHA_ORDER (una vez)...")
    con = sqlite3.connect(META)
    n_train = con.execute("SELECT COUNT(*) FROM meta WHERE rl_fs_t <= ?", (TRAIN_SPLIT,)).fetchone()[0]
    print(f"  filas train oficiales (con missing): {n_train}")
    # shas en orden rowid y timestamps en orden rowid
    # Usar memmap temporal en WORK (20GB disponible)
    ORDER = os.path.join(WORK, "tmp_all_shas.npy")
    TS_ALL = os.path.join(WORK, "tmp_all_ts.npy")
    mm_shas = np.memmap(ORDER, dtype="S64", mode="w+", shape=(n_train,))
    mm_ts = np.memmap(TS_ALL, dtype="f8", mode="w+", shape=(n_train,))
    pos=0
    for sha, ts in con.execute("SELECT sha256, rl_fs_t FROM meta WHERE rl_fs_t <= ? ORDER BY rowid", (TRAIN_SPLIT,)):
        mm_shas[pos]=sha.encode(); mm_ts[pos]=float(ts); pos+=1
    mm_shas.flush(); mm_ts.flush()
    assert pos==n_train
    # filtrar missing
    miss_arr=np.array(sorted(missing), dtype="S64")
    keep=np.ones(n_train, dtype=bool)
    CH=1000000
    for a in range(0,n_train,CH):
        keep[a:a+CH]=~np.isin(mm_shas[a:a+CH], miss_arr)
    n_keep=int(keep.sum())
    assert n_keep==EXPECTED_NPZ_ROWS
    # unsorted filtrados
    shas_unsorted=np.memmap(os.path.join(WORK,"tmp_filt_shas.npy"), dtype="S64", mode="w+", shape=(n_keep,))
    ts_unsorted=np.memmap(os.path.join(WORK,"tmp_filt_ts.npy"), dtype="f8", mode="w+", shape=(n_keep,))
    w=0
    for a in range(0,n_train,CH):
        blk_shas=mm_shas[a:a+CH][keep[a:a+CH]]
        blk_ts=mm_ts[a:a+CH][keep[a:a+CH]]
        shas_unsorted[w:w+len(blk_shas)]=blk_shas
        ts_unsorted[w:w+len(blk_ts)]=blk_ts
        w+=len(blk_shas)
    shas_unsorted.flush(); ts_unsorted.flush()
    # ordenar por sha para alinear con SHA_ORDER
    order=np.argsort(shas_unsorted)
    # verificar que el orden coincide con SHA_ORDER existente
    # (comparar 3 puntos + hash muestral)
    assert np.array_equal(shas_unsorted[order[:3]], shas[:3])
    assert np.array_equal(shas_unsorted[order[-3:]], shas[-3:])
    print("  orden verificado contra SHA_ORDER existente")
    # escribir timestamps ordenados
    ts_sorted=np.memmap(TS_SORTED, dtype="f8", mode="w+", shape=(n_keep,))
    CH2=2000000
    for a in range(0,n_keep,CH2):
        ts_sorted[a:a+CH2]=ts_unsorted[order[a:a+CH2]]
    ts_sorted.flush()
    # limpiar temporales
    for f in [ORDER, TS_ALL, os.path.join(WORK,"tmp_filt_shas.npy"), os.path.join(WORK,"tmp_filt_ts.npy")]:
        try: os.remove(f)
        except: pass
    con.close()
    print(f"  artefacto TS_SORTED: {os.path.getsize(TS_SORTED)/2**20:.1f} MB")
else:
    print(f"Artefacto TS_SORTED existente: {os.path.getsize(TS_SORTED)/2**20:.1f} MB")

ts = np.memmap(TS_SORTED, dtype="f8", mode="r")
print(f"rl_fs_t rango: [{ts.min():.0f}, {ts.max():.0f}]  (span {(ts.max()-ts.min())/86400:.1f} días)")
# convertir a datetime para legibilidad
import datetime
def to_date(x): return datetime.datetime.utcfromtimestamp(float(x)).strftime("%Y-%m-%d")
print(f"  fecha min: {to_date(ts.min())}  max: {to_date(ts.max())}")
print(f"  mediana: {to_date(np.median(ts))}  p10: {to_date(np.quantile(ts,0.1))}  p90: {to_date(np.quantile(ts,0.9))}")
# exclusiones
print(f"Exclusiones: missing={len(missing)}, fuera TRAIN (>={TRAIN_SPLIT:.0f}) no incluidos por query, valid/test no tocados")
# RAM
proc=psutil.Process()
print(f"RAM usada tras inventario: {proc.memory_info().rss/2**30:.2f} GB  | tiempo: {time.time()-t0:.1f}s")


In [ ]:
import time as _t
t0 = _t.time()
# FASE 2C — Muestreo estratificado 7M (seed fija, proporciones preservadas)
import numpy as np, os, time, psutil, sqlite3
SEED = 42
TARGET = 7_000_000
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
SHA_ORDER = os.path.join(WORK, "train_sha_order.npy")
LAB_SURV = os.path.join(WORK, "train_lab_surv.npy")
TS_SORTED = os.path.join(WORK, "train_ts_sorted.npy")
TRAIN_SPLIT = 1543542570.0
VAL_SPLIT = 1547279640.0

shas = np.memmap(SHA_ORDER, dtype="S64", mode="r")
labs = np.memmap(LAB_SURV, dtype="i1", mode="r")
ts = np.memmap(TS_SORTED, dtype="f8", mode="r")
N = shas.shape[0]
assert N==12699013

# Bins deciles (re-derivados determinísticamente)
qs = np.linspace(0,1,11)
bin_edges = np.quantile(ts, qs)
bins = np.digitize(ts, bin_edges[1:-1])
N_BINS=10

# estratos: (label, bin) -> 20 estratos
rng = np.random.default_rng(SEED)
# Pre-calcular índices por estrato (lista de arrays de posiciones)
stratum_indices = {}
for lab in [0,1]:
    for b in range(N_BINS):
        idx = np.where((labs==lab) & (bins==b))[0]
        stratum_indices[(lab,b)] = idx

# tamaño original por estrato
orig_counts = {k: len(v) for k,v in stratum_indices.items()}
total = N
t0=time.time()
# Asignación proporcional con corrección de redondeo
raw_alloc = {k: orig_counts[k]/total * TARGET for k in orig_counts}
alloc = {k: int(round(v)) for k,v in raw_alloc.items()}
# Ajuste para sumar exactamente TARGET
diff = TARGET - sum(alloc.values())
if diff!=0:
    # ordenar estratos por parte fraccional descendente (si sobra) o ascendente (si falta)
    frac = {k: raw_alloc[k]-alloc[k] for k in alloc}
    order = sorted(frac, key=lambda k: frac[k], reverse=(diff>0))
    for i in range(abs(diff)):
        k = order[i % len(order)]
        alloc[k] += 1 if diff>0 else -1
assert sum(alloc.values())==TARGET
# clamp a tamaño del estrato (no sobremuestrear)
for k in list(alloc):
    if alloc[k] > orig_counts[k]:
        alloc[k]=orig_counts[k]
# re-ajustar si hubo clamp
if sum(alloc.values())!=TARGET:
    # repartir el delta entre estratos con capacidad
    remaining = TARGET - sum(alloc.values())
    # capacidad = orig - alloc
    caps = {k: orig_counts[k]-alloc[k] for k in alloc}
    order = sorted(caps, key=lambda k: caps[k], reverse=True)
    for k in order:
        if remaining==0: break
        add = min(caps[k], remaining if remaining>0 else 0)
        if remaining<0:
            add = -min(alloc[k], -remaining)
        alloc[k]+=add; remaining-=add

assert sum(alloc.values())==TARGET
assert all(alloc[k]<=orig_counts[k] for k in alloc)

# Muestreo sin reemplazo por estrato
selected_lists=[]
for k, idx in stratum_indices.items():
    n = alloc[k]
    if n==0: continue
    # choice sin reemplazo, seed determinística vía rng
    sel = rng.choice(idx, size=n, replace=False)
    selected_lists.append(sel)
selected = np.concatenate(selected_lists)
# barajar globalmente para no dejar bloques por estrato (orden aleatorio pero reproducible)
rng.shuffle(selected)
assert selected.shape==(TARGET,) and len(np.unique(selected))==TARGET
# Verificar preservación de distribución
from collections import Counter
lab_sel = labs[selected]
n_mal_sel=int((lab_sel==1).sum()); n_ben_sel=int((lab_sel==0).sum())
print(f"SEED={SEED}  TARGET={TARGET}")
print(f"Original: malware {orig_counts[(1,0)]+ sum(orig_counts[(1,b)] for b in range(1,10))}?? total mal {sum(orig_counts[(1,b)] for b in range(10))} ben {sum(orig_counts[(0,b)] for b in range(10))}")
# mejor: totales
orig_mal = int((labs==1).sum()); orig_ben=int((labs==0).sum())
print(f"  Original TRAIN: malware {orig_mal} ({orig_mal/N:.2%})  benigno {orig_ben} ({orig_ben/N:.2%})")
print(f"  Seleccionado 7M: malware {n_mal_sel} ({n_mal_sel/TARGET:.2%})  benigno {n_ben_sel} ({n_ben_sel/TARGET:.2%})  delta mal {n_mal_sel/TARGET - orig_mal/N:+.2%}")
# distribución temporal resultante
bins_sel = bins[selected]
hist_sel = np.zeros((2, N_BINS), dtype=int)
for lab in [0,1]:
    hist_sel[lab]=np.bincount(bins_sel[lab_sel==lab], minlength=N_BINS)[:N_BINS]
print("\nDistribución resultante 7M por bin temporal (deciles):")
for lab in [0,1]:
    name="benigno" if lab==0 else "malware"
    print(f"  {name:8s}:", " ".join(f"{c:6d}" for c in hist_sel[lab]))

# Guardar índices
OUT_IDX = os.path.join(WORK, "train_7m_indices.npy")
np.save(OUT_IDX, selected.astype(np.uint32))
print(f"\nÍndices guardados: {OUT_IDX}  shape {selected.shape} dtype {selected.dtype}  {os.path.getsize(OUT_IDX)/2**20:.1f} MB")

# Guardar manifest Parquet
import pandas as pd
shas_sel = shas[selected].astype(str)  # S64 -> str
labs_sel = lab_sel.astype(np.int8)
ts_sel = ts[selected]
# row_idx es la posición en NPZ (que es el índice seleccionado)
row_idx = selected.astype(np.uint32)
# Verificar correspondencia row_idx <-> sha (ya ordenado)
assert np.all(shas[row_idx].astype(str)==shas_sel)

OUT_PARQ = os.path.join(WORK, "train_7m_manifest.parquet")
df = pd.DataFrame({"row_idx": row_idx, "sha256": shas_sel, "is_malware": labs_sel, "rl_fs_t": ts_sel})
# tipos eficientes
df["is_malware"]=df["is_malware"].astype("int8")
df["row_idx"]=df["row_idx"].astype("uint32")
df["rl_fs_t"]=df["rl_fs_t"].astype("float64")
df.to_parquet(OUT_PARQ, compression="zstd", index=False)
print(f"Manifest Parquet: {OUT_PARQ}  {os.path.getsize(OUT_PARQ)/2**20:.1f} MB  filas {len(df)}")
# preview CSV pequeño
preview = os.path.join(WORK, "train_7m_preview.csv")
df.head(20).to_csv(preview, index=False)
print(f"Preview CSV (20 filas): {preview}")

# Asserts fuertes
import sqlite3, json
con=sqlite3.connect(os.path.join(WORK,"meta.db"))
# 1. exactamente 7M
assert len(df)==TARGET
# 2-3. índices y shas únicos
assert df["row_idx"].nunique()==TARGET
assert df["sha256"].nunique()==TARGET
# 4. ninguna fuera TRAIN
assert df["rl_fs_t"].max() <= TRAIN_SPLIT + 1e-6
assert df["rl_fs_t"].min() >= df["rl_fs_t"].min()  # trivial, pero valida no NaN
assert df["rl_fs_t"].isna().sum()==0
# 5. ningún sha en missing
missing=set(json.load(open(os.path.join(WORK,"shas_missing_ember_features.json"))))
assert not any(s in missing for s in df["sha256"].head(1000).tolist() )  # muestral rápido
# full check por isin con memmap
miss_arr=np.array(sorted(missing), dtype="S64")
# check 7M no solape con missing via isin chunked
CH=500000
for a in range(0, TARGET, CH):
    blk=np.array(df["sha256"].iloc[a:a+CH].tolist(), dtype="S64")
    assert not np.isin(blk, miss_arr).any(), f"sha missing en bloque {a}"
# 6. labels válidos
assert set(df["is_malware"].unique()) <= {0,1}
# 7. correspondencia row_idx<->sha consistente con SHA_ORDER
shas_full=np.memmap(SHA_ORDER, dtype="S64", mode="r")
for a in range(0, min(TARGET, 10000), 2000):
    assert shas_full[df["row_idx"].iloc[a]]==df["sha256"].iloc[a].encode()
# 8. ningún solapamiento valid/test (verificar que ningún sha aparece con rl_fs_t en valid/test)
# ya garantizado por rl_fs_t <= TRAIN_SPLIT, pero verificamos contra DB que no hay shas de valid
val_shas=set(r[0] for r in con.execute("SELECT sha256 FROM meta WHERE rl_fs_t > ? LIMIT 1000", (TRAIN_SPLIT,)))
assert df["sha256"].isin(val_shas).sum()==0
print("\nAsserts fuertes: PASS")
print(f"Tiempo muestreo+guardado: {time.time()-t0:.1f}s  RAM {psutil.Process().memory_info().rss/2**30:.2f} GB")


In [ ]:
# FASE 2D — Puerta contra el dataset aprobado (numeros de Fase 2 validada).
import numpy as np, pandas as pd, os
MANIFEST = os.path.join(WORK, "train_7m_manifest.parquet")
INDICES = os.path.join(WORK, "train_7m_indices.npy")
APPROVED_MAL, APPROVED_BEN = 4187321, 2812679  # 59.82% / 40.18%, seed 42
df = pd.read_parquet(MANIFEST)
idx = np.load(INDICES)
assert len(df) == 7_000_000 and idx.shape == (7_000_000,) and idx.dtype == np.uint32
assert list(df.columns) == ["row_idx", "sha256", "is_malware", "rl_fs_t"], list(df.columns)
n_mal = int((df["is_malware"] == 1).sum())
n_ben = int((df["is_malware"] == 0).sum())
print(f"manifest: malware={n_mal} benigno={n_ben}")
assert (n_mal, n_ben) == (APPROVED_MAL, APPROVED_BEN), "FAIL: distribucion != aprobada"
assert df["row_idx"].max() < EXPECTED_NPZ_ROWS and df["row_idx"].min() >= 0
assert df["rl_fs_t"].max() <= TRAIN_SPLIT + 1e-6, "FAIL: leakage temporal"
print("PASS Fase 2: manifest 7M == dataset aprobado (seed 42, sin leakage).")


In [ ]:
# FASE 3A — Infra streaming: planificador de bloques + fetch validado + chunk size.
# A 55.1% de densidad, fusionar rangos degenera en transferencia casi total: se documenta
# y se mide honestamente (requests, bytes, MB/s). El orden de partial_fit no es relevante.
import time, psutil
import numpy as np

TARGET = 7_000_000
INDICES = os.path.join(WORK, "train_7m_indices.npy")
idx = np.load(INDICES)
assert idx.shape == (TARGET,) and idx.dtype == np.uint32
sel = np.sort(idx)  # copia ordenada solo para planificar I/O (56MB)

vm = psutil.virtual_memory()
print(f"RAM total={vm.total/2**30:.1f}GB disponible={vm.available/2**30:.1f}GB")
# Cap conservador: fetch <=256MB y buffer partial_fit <=~320MB, sin usar RAM de mas.
FETCH_SPAN_ROWS = 25000   # ~238MB por Range GET
PARTIAL_ROWS = 32768      # ~312MB por partial_fit
MAX_GAP_ROWS = 512        # fusiona huecos pequenos dentro de un span
print(f"FETCH_SPAN_ROWS={FETCH_SPAN_ROWS} (~{FETCH_SPAN_ROWS*ROW_BYTES/2**20:.0f}MB/req)")
print(f"PARTIAL_ROWS={PARTIAL_ROWS} (~{PARTIAL_ROWS*ROW_BYTES/2**20:.0f}MB/partial_fit)")

def plan_spans(s, max_gap=MAX_GAP_ROWS, max_span=FETCH_SPAN_ROWS):
    # Agrupa indices ordenados en spans [start, end) fusionando huecos <= max_gap
    # y cortando spans que excedan max_span. Devuelve lista de (start, end).
    spans = []
    start = prev = int(s[0])
    for v in s[1:]:
        v = int(v)
        if v - prev - 1 <= max_gap and v - start + 1 <= max_span:
            prev = v
            continue
        spans.append((start, prev + 1))
        start = prev = v
    spans.append((start, prev + 1))
    return spans

def fetch_span(s, e):
    # Un Range GET -> (e-s, 2381) float32 con validacion dura por chunk.
    d = npz_range(PAY0 + s * ROW_BYTES, PAY0 + e * ROW_BYTES - 1)
    X = np.frombuffer(d, dtype=np.float32).reshape(-1, N_FEATURES)
    assert X.ndim == 2 and X.shape[1] == N_FEATURES, f"FAIL chunk dim: {X.shape}"
    assert X.dtype == np.float32, f"FAIL chunk dtype: {X.dtype}"
    assert np.all(np.isfinite(X)), "FAIL chunk: NaN/Inf — abortar"
    return X

spans = plan_spans(sel)
span_rows = sum(e - s for s, e in spans)
print(f"spans={len(spans)} filas_span={span_rows} seleccionadas={TARGET} "
      f"redundancia={span_rows/TARGET:.2f}x transferencia_est={span_rows*ROW_BYTES/2**30:.1f}GB")
print("PASS Fase 3A: planificador listo.")


In [ ]:
# FASE 3B — StandardScaler streaming: spans -> mascara -> partial_fit.
from sklearn.preprocessing import StandardScaler
import psutil
scaler = StandardScaler()
io_stats = {"requests": 0, "bytes": 0, "rows_fetched": 0, "rows_selected": 0}
max_rss = 0.0
t_scaler = time.time()
buf, buf_rows, seen = [], 0, 0

def _rss():
    return psutil.Process().memory_info().rss / 2**30

for k, (s, e) in enumerate(spans):
    X = fetch_span(s, e)
    io_stats["requests"] += 1
    io_stats["bytes"] += (e - s) * ROW_BYTES
    io_stats["rows_fetched"] += (e - s)
    lo = int(np.searchsorted(sel, s))
    hi = int(np.searchsorted(sel, e))
    Xsel = X[sel[lo:hi] - s]
    io_stats["rows_selected"] += Xsel.shape[0]
    buf.append(Xsel)
    buf_rows += Xsel.shape[0]
    while buf_rows >= PARTIAL_ROWS:
        big = np.concatenate(buf, axis=0)
        use, rest = big[:PARTIAL_ROWS], big[PARTIAL_ROWS:]
        assert use.shape == (PARTIAL_ROWS, N_FEATURES) and np.all(np.isfinite(use))
        scaler.partial_fit(use)
        seen += use.shape[0]
        buf = [rest] if rest.shape[0] else []
        buf_rows = rest.shape[0]
        del big, use, rest
    max_rss = max(max_rss, _rss())
    if (k + 1) % 25 == 0 or (k + 1) == len(spans):
        el = time.time() - t_scaler
        print(f"  span {k+1}/{len(spans)} seen={seen} MB={io_stats['bytes']/2**20:.0f} "
              f"MB/s={io_stats['bytes']/2**20/max(el,1e-6):.1f} RSS={_rss():.2f}GB", flush=True)
if buf_rows:
    tail = np.concatenate(buf, axis=0)
    assert tail.shape[1] == N_FEATURES and np.all(np.isfinite(tail))
    scaler.partial_fit(tail)
    seen += tail.shape[0]
    del tail
    buf, buf_rows = [], 0
t_scaler_total = time.time() - t_scaler
max_rss = max(max_rss, _rss())
print(f"partial_fit completo: seen={seen} requests={io_stats['requests']} "
      f"GB={io_stats['bytes']/2**30:.2f} tiempo={t_scaler_total:.1f}s RSSmax={max_rss:.2f}GB")
assert io_stats["rows_selected"] == TARGET, "FAIL: filas seleccionadas != 7M"
assert scaler.n_samples_seen_ == TARGET, f"FAIL: n_samples_seen_={scaler.n_samples_seen_}"
print("PASS Fase 3B: scaler ajustado sobre 7M exactas.")


In [ ]:
# FASE 3C — Auditoria, comparativa opcional, persistencia, test 1000 y veredicto.
import hashlib, json, pickle
from datetime import datetime, timezone
import numpy as np

# --- auditoria del scaler ---
assert scaler.n_samples_seen_ == 7_000_000
mean_, var_, scale_ = scaler.mean_, scaler.var_, scaler.scale_
assert mean_.shape == (N_FEATURES,) and var_.shape == (N_FEATURES,) and scale_.shape == (N_FEATURES,)
assert np.all(np.isfinite(mean_)) and np.all(np.isfinite(var_)) and np.all(np.isfinite(scale_))
near_zero = int((var_ < 1e-12).sum())
print(f"mean_[:5]={np.array2string(mean_[:5], precision=4)} mean_[-5:]={np.array2string(mean_[-5:], precision=4)}")
print(f"var_ min={var_.min():.3g} max={var_.max():.3g} | scale_ min={scale_.min():.3g} max={scale_.max():.3g}")
print(f"features varianza~=0 (var<1e-12): {near_zero}")

# --- comparativa con scaler actual de Shadow-Net (solo si esta disponible; nunca se modifica) ---
import glob
cands = glob.glob("/kaggle/input/**/scaler*.pkl", recursive=True)
if cands:
    cur = pickle.load(open(cands[0], "rb"))
    print(f"scaler actual: {cands[0]} n_features={cur.mean_.shape[0]}")
    if cur.mean_.shape == mean_.shape:
        dmean = np.abs(cur.mean_ - mean_)
        with np.errstate(divide="ignore", invalid="ignore"):
            dscale = np.abs(cur.scale_ - scale_) / np.maximum(scale_, 1e-12)
        top = np.argsort(dmean)[-5:][::-1]
        print(f"  delta_abs mean_: mediana={np.median(dmean):.3g} max={dmean.max():.3g} top5_idx={top.tolist()}")
        print(f"  delta_rel scale_: mediana={np.median(dscale):.3g} max={dscale.max():.3g}")
    else:
        print("  dims distintas: comparativa omitida (no bloqueante)")
else:
    print("scaler actual no disponible en Kaggle: comparativa omitida (no bloqueante)")

# --- persistencia ---
PKL = os.path.join(WORK, "scaler_sorel_7m_v1.1.pkl")
JSN = os.path.join(WORK, "scaler_sorel_7m_v1.1.json")
with open(PKL, "wb") as f:
    pickle.dump(scaler, f, protocol=4)
sha_pkl = hashlib.sha256(open(PKL, "rb").read()).hexdigest()
meta = {
    "version": "scaler_sorel_7m_v1.1",
    "dataset": "SOREL-20M train split oficial, seleccion 7M seed 42 (59.82% malware)",
    "n_samples": 7_000_000,
    "n_features": N_FEATURES,
    "seed": 42,
    "method": "StandardScaler.partial_fit streaming por spans Range (STORED float32)",
    "chunk_size": {"fetch_span_rows": FETCH_SPAN_ROWS, "partial_rows": PARTIAL_ROWS, "max_gap_rows": MAX_GAP_ROWS},
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "sha256_pkl": sha_pkl,
    "versiones": {"numpy": np.__version__, "sklearn": __import__("sklearn").__version__, "python": sys.version.split()[0]},
    "io": {**io_stats, "gb_transferidos": io_stats["bytes"] / 2**30},
    "tiempo_total_s": t_scaler_total,
    "ram_max_gb": max_rss,
    "auditoria": {"near_zero_var": near_zero, "scale_min": float(scale_.min()), "scale_max": float(scale_.max())},
}
open(JSN, "w").write(json.dumps(meta, indent=2))
assert os.path.getsize(PKL) > 0 and os.path.getsize(JSN) > 0
print(f"artefactos: {PKL} ({os.path.getsize(PKL)/1024:.1f}KB, sha256={sha_pkl[:16]}...) {JSN}")
# recarga y verificacion post-guardado
sc2 = pickle.load(open(PKL, "rb"))
assert sc2.n_samples_seen_ == 7_000_000 and np.array_equal(sc2.mean_, mean_)
print("recarga post-guardado: OK")

# --- test 1000 muestras (sanity, no 0/1 exactos) ---
rng_t = np.random.default_rng(12345)
tpos = np.sort(rng_t.choice(TARGET, 1000, replace=False))
trows = np.sort(idx[tpos])
tspans = plan_spans(trows, max_gap=2048, max_span=50000)
parts = []
for s, e in tspans:
    X = fetch_span(s, e)
    lo = int(np.searchsorted(trows, s)); hi = int(np.searchsorted(trows, e))
    parts.append(X[trows[lo:hi] - s])
Xt = np.concatenate(parts, axis=0)
assert Xt.shape == (1000, N_FEATURES)
mean_before = scaler.mean_.copy()
seen_before = scaler.n_samples_seen_
Zt = scaler.transform(Xt)
assert Zt.shape == (1000, N_FEATURES) and np.all(np.isfinite(Zt))
assert scaler.n_samples_seen_ == seen_before and np.array_equal(scaler.mean_, mean_before), "FAIL: scaler mutado"
print(f"test 1000: shape={Zt.shape} finite=True scaler_no_mutado=True (sanity, sin exigir media 0/1)")
print("STANDARD SCALER 2381: PASS")


## Fase 4 — Overlay feature engineering (RESERVADA)
Definir las N features de overlay (`X = [EMBER_2381 | OVERLAY_N]`) y su pipeline. No implementar en esta ejecucion.

## Fase 5 — Streaming Dataset / DataLoader (RESERVADA)
`torch.utils.data.IterableDataset` sobre `train_7m_indices.npy` + Range. No implementar.

## Fase 6 — FFNN PyTorch (RESERVADA)
Arquitectura MLP compatible con Shadow-Net + export ONNX futuro. No implementar.

## Fase 7 — Training (RESERVADA)
Loop con AMP/grad-clip/scheduler/checkpoints. No implementar.

## Fase 8 — Validation / metrics (RESERVADA)
AUC, calibracion, curvas. No implementar.

## Fase 9 — Ablation / threshold calibration (RESERVADA)
No implementar.

## Fase 10 — ONNX export (RESERVADA)
`torch.onnx.export` + validacion con onnxruntime. No implementar.

STOP — esta ejecucion termina en Fase 3.
